In [ ]:
# Thiết lập thư mục gốc một lần để notebook chạy được cả trên máy cá nhân và Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import copy
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import sys

# Set seed cố định
def set_seed(s=42):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  torch.cuda.manual_seed_all(s)
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")

sys.path.append(str(DATA_DIR / "src/eval")) 
from evaluator import evaluate

Sử dụng thiết bị: cuda


In [ ]:
# 1. Nạp tập train kịch bản warm
tr_warm = pd.read_parquet(str(DATA_DIR / "data/splits/warm_scenario/train_warm.parquet"))
va_warm = pd.read_parquet(str(DATA_DIR / "data/splits/warm_scenario/val_warm.parquet"))

# 2. Trích xuất và sắp xếp danh sách user_id độc lập
warm_users = sorted(tr_warm['user_id'].astype(str).unique().tolist())
user2idx = {str(u): i for i, u in enumerate(warm_users)}
num_users = len(user2idx)

# 3. Lưu thành các tệp JSON chỉ mục chuẩn cho kịch bản warm
with open('user_index_warm.json', 'w') as f:
  json.dump(warm_users, f)

with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/item_index.json"), 'r') as f:
  course_ids = json.load(f)

assert len(course_ids) == 3769, f"Bảng chỉ số phải có đúng 3769 khóa, nhận {len(course_ids)}"
item2idx = {str(c): i for i, c in enumerate(course_ids)}
num_items = len(item2idx)

print(f"Tổng số User: {num_users}, Tổng số Item: {num_items}")

Tổng số User: 88729, Tổng số Item: 3769


In [4]:
u_all = tr_warm['user_id'].astype(str).map(user2idx).to_numpy()
i_all = tr_warm['course_id'].astype(str).map(item2idx).to_numpy()

warm_idx = torch.tensor(np.unique(i_all), dtype=torch.long)

# Xây dựng val_target_dict từ val_warm
val_target_dict = {}
for u, group in va_warm.groupby('user_id'):
  u_str = str(u)
  if u_str in user2idx:
    u_idx = user2idx[u_str]
    item_idxs = [item2idx[str(i)] for i in group['course_id'] if str(i) in item2idx]
    if len(item_idxs) > 0:
      val_target_dict[u_idx] = item_idxs

# Lịch sử train warm để mask khi evaluate val
train_hist_dict = {}
for u, i in zip(u_all, i_all):
  u = int(u)
  if u not in train_hist_dict:
    train_hist_dict[u] = set()
  train_hist_dict[u].add(int(i))

In [5]:
# 3. Xây dựng ma trận kề cho train_warm
row = torch.tensor(u_all, dtype=torch.long)
col = torch.tensor(i_all, dtype=torch.long) + num_users
row_full = torch.cat([row, col])
col_full = torch.cat([col, row])
val_full = torch.ones(row_full.size(0), dtype=torch.float32)

assoc = torch.sparse_coo_tensor(
    torch.stack([row_full, col_full]),
    val_full,
    (num_users + num_items, num_users + num_items),
).to(DEVICE)
deg = torch.sparse.sum(assoc, dim=1).to_dense()
deg_inv_sqrt = torch.pow(deg, -0.5)
deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0.0

i_idx = torch.arange(num_users + num_items, device=DEVICE)
diag_indices = torch.stack([i_idx, i_idx], dim=0)

deg_mat_inv_sqrt = torch.sparse_coo_tensor(
    diag_indices,
    deg_inv_sqrt.to(DEVICE),
    (num_users + num_items, num_users + num_items),
)

adj_mat_warm = torch.sparse.mm(deg_mat_inv_sqrt, torch.sparse.mm(assoc, deg_mat_inv_sqrt))

/tmp/ipykernel_23/2460344631.py:8: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  assoc = torch.sparse_coo_tensor(


In [6]:
# 5. Định nghĩa mô hình LightGCN
class LightGCN(nn.Module):

  def __init__(self, num_users, num_items, latent_dim=64, n_layers=3):
    super(LightGCN, self).__init__()
    self.num_users = num_users
    self.num_items = num_items
    self.user_embedding = nn.Embedding(num_users, latent_dim)
    self.item_embedding = nn.Embedding(num_items, latent_dim)
    nn.init.xavier_uniform_(self.user_embedding.weight)
    nn.init.xavier_uniform_(self.item_embedding.weight)

  def forward(self, adj_matrix):
    users_emb = self.user_embedding.weight
    items_emb = self.item_embedding.weight
    all_emb = torch.cat([users_emb, items_emb], dim=0)
    embs = [all_emb]
    for _ in range(3):
      all_emb = torch.sparse.mm(adj_matrix, all_emb)
      embs.append(all_emb)
    lightgcn_emb = torch.stack(embs, dim=1).mean(dim=1)
    return torch.split(lightgcn_emb, [self.num_users, self.num_items])

def bpr_loss(users, pos_items, neg_items, u_emb, i_emb):
  u_g = u_emb[users]
  pos_g = i_emb[pos_items]
  neg_g = i_emb[neg_items]
  pos_scores = torch.sum(u_g * pos_g, dim=1)
  neg_scores = torch.sum(u_g * neg_g, dim=1)
  return -torch.mean(torch.nn.functional.logsigmoid(pos_scores - neg_scores))


In [7]:
# 6. Huấn luyện LightGCN Warm
LATENT_DIM = 64
N_LAYERS = 3
BATCH_SIZE = 1024
LR = 1e-3
EPOCHS = 60

model = LightGCN(num_users, num_items, latent_dim=LATENT_DIM, n_layers=N_LAYERS).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

best_recall = 0.0
best_state = None

u_tensor = torch.tensor(u_all, dtype=torch.long)
i_tensor = torch.tensor(i_all, dtype=torch.long)

print('--- Bắt đầu huấn luyện LightGCN Warm Backbone ---')
for epoch in range(EPOCHS):
  model.train()
  total_loss = 0
  perm = torch.randperm(len(u_tensor))
  sampled_u = u_tensor[perm]
  sampled_pos_i = i_tensor[perm]
  sampled_neg_i = warm_idx[
      torch.randint(0, len(warm_idx), (len(perm),), dtype=torch.long)
  ]  # Mẫu âm chỉ trong khóa có trong train warm

  for s in range(0, len(perm), BATCH_SIZE):
    batch_u = sampled_u[s : s + BATCH_SIZE].to(DEVICE)
    batch_pos = sampled_pos_i[s : s + BATCH_SIZE].to(DEVICE)
    batch_neg = sampled_neg_i[s : s + BATCH_SIZE].to(DEVICE)

    optimizer.zero_grad()
    u_emb, i_emb = model(adj_mat_warm)
    loss = bpr_loss(batch_u, batch_pos, batch_neg, u_emb, i_emb)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()

  # Đánh giá trên val_warm
  model.eval()
  with torch.no_grad():
    u_emb, i_emb = model(adj_mat_warm)


    def score_fn(users_tensor):
      return torch.matmul(u_emb[users_tensor.to(DEVICE)], i_emb.t())


    val_res = evaluate(
        score_fn, val_target_dict, mask=train_hist_dict, ks=(20,), bs=1024
    )
    current_recall = val_res['Recall@20'].mean()

  print(f'Epoch {epoch+1:02d}/{EPOCHS} | Loss: {total_loss:.4f} | Recall@20 (Val_Warm): {current_recall:.4f}')

  if current_recall > best_recall:
    best_recall = current_recall
    best_state = copy.deepcopy(model.state_dict())


print(f"\nHuấn luyện hoàn thành! Best Recall@20: {best_recall:.4f}")

--- Bắt đầu huấn luyện LightGCN Warm Backbone ---
Epoch 01/60 | Loss: 121.0084 | Recall@20 (Val_Warm): 0.0858
Epoch 02/60 | Loss: 76.0659 | Recall@20 (Val_Warm): 0.0959
Epoch 03/60 | Loss: 64.0233 | Recall@20 (Val_Warm): 0.1039
Epoch 04/60 | Loss: 57.5114 | Recall@20 (Val_Warm): 0.1098
Epoch 05/60 | Loss: 52.1002 | Recall@20 (Val_Warm): 0.1181
Epoch 06/60 | Loss: 48.2401 | Recall@20 (Val_Warm): 0.1275
Epoch 07/60 | Loss: 45.0085 | Recall@20 (Val_Warm): 0.1312
Epoch 08/60 | Loss: 42.1720 | Recall@20 (Val_Warm): 0.1363
Epoch 09/60 | Loss: 40.0209 | Recall@20 (Val_Warm): 0.1362
Epoch 10/60 | Loss: 37.7699 | Recall@20 (Val_Warm): 0.1389
Epoch 11/60 | Loss: 36.1768 | Recall@20 (Val_Warm): 0.1440
Epoch 12/60 | Loss: 34.6905 | Recall@20 (Val_Warm): 0.1442
Epoch 13/60 | Loss: 33.1923 | Recall@20 (Val_Warm): 0.1478
Epoch 14/60 | Loss: 31.6787 | Recall@20 (Val_Warm): 0.1501
Epoch 15/60 | Loss: 30.2381 | Recall@20 (Val_Warm): 0.1490
Epoch 16/60 | Loss: 29.5352 | Recall@20 (Val_Warm): 0.1498
Epoch

In [8]:
# 7. Xuất ma trận embedding warm chuẩn
# 6. Lưu output g_u_warm.npy và g_i_warm.npy
model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
  u_emb_final, i_emb_final = model(adj_mat_warm)
  g_u_warm = u_emb_final.cpu().numpy()
  g_i_warm = i_emb_final.cpu().numpy()

assert g_u_warm.shape == (num_users, 64), f"Lỗi shape g_u: {g_u_warm.shape}"

np.save('g_u_warm.npy', g_u_warm)
np.save('g_i_warm.npy', g_i_warm)

print(f'Ma trận lưu thành công: g_u_warm.npy ({g_u_warm.shape}) và g_i_warm.npy ({g_i_warm.shape})')

md5_res = {p: hashlib.md5(open(p, "rb").read()).hexdigest() for p in ('g_u_warm.npy', 'g_i_warm.npy')}
print("Xuất file hoàn tất thành công!")
print(f"Bảng kích thước: g_u_warm = {g_u_warm.shape}, g_i_warm = {g_i_warm.shape}")
print(f"MD5 Checksum: {md5_res}")

Ma trận lưu thành công: g_u_warm.npy ((88729, 64)) và g_i_warm.npy ((3769, 64))
Xuất file hoàn tất thành công!
Bảng kích thước: g_u_warm = (88729, 64), g_i_warm = (3769, 64)
MD5 Checksum: {'g_u_warm.npy': '75de5e78b19a66dfb3135a604fa839ca', 'g_i_warm.npy': '0bd597eddf25f5c58f818f8b6b07f57a'}
